<a href="https://colab.research.google.com/github/branka2004/preporuka_proizvoda_u_elektronskoj_trgovini/blob/main/preporuka_proizvoda_autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Primena deep learninga za preporuku proizvoda u elektronskoj trgovini pomoću autoenkodera

## Projekat iz predmeta: Duboko učenje i neuronske mreže

### Autori:
- Nikola Ilic 2022/0190
- Branka Bakovic 2022/0295

# Opis projekta

Cilj ovog projekta je implementacija sistema za preporuku proizvoda korišćenjem tehnika dubokog učenja i autoenkodera.

Model će biti treniran nad Amazon Reviews skupom podataka kako bi naučio latentne reprezentacije korisničkih preferencija i generisao personalizovane preporuke proizvoda.

U okviru projekta biće analizirane različite arhitekture autoenkodera, performanse modela i kvalitet preporuka korišćenjem standardnih evaluacionih metrika.

# Neophodni importi

In [14]:
import io
import os
import random

import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns


import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

RAND_STATE = 42

print("PyTorch verzija:", torch.__version__)
print("Da li je GPU dostupan:", torch.cuda.is_available())

PyTorch verzija: 2.11.0+cu128
Da li je GPU dostupan: True


In [15]:
df = pd.read_csv('/content/Reviews.csv')

df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   Id                      568454 non-null  int64 
 1   ProductId               568454 non-null  object
 2   UserId                  568454 non-null  object
 3   ProfileName             568428 non-null  object
 4   HelpfulnessNumerator    568454 non-null  int64 
 5   HelpfulnessDenominator  568454 non-null  int64 
 6   Score                   568454 non-null  int64 
 7   Time                    568454 non-null  int64 
 8   Summary                 568427 non-null  object
 9   Text                    568454 non-null  object
dtypes: int64(5), object(5)
memory usage: 43.4+ MB


# Analiza skupa podataka

In [17]:
print("Broj redova i kolona:")
print(df.shape)

print("\nNazivi kolona:")
print(df.columns)

print("\nBroj jedinstvenih korisnika:")
print(df['UserId'].nunique())

print("\nBroj jedinstvenih proizvoda:")
print(df['ProductId'].nunique())

print("Nedostajuće vrednosti:\n")
print(df.isnull().sum())



Broj redova i kolona:
(568454, 10)

Nazivi kolona:
Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

Broj jedinstvenih korisnika:
256059

Broj jedinstvenih proizvoda:
74258
Nedostajuće vrednosti:

Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64
